# 10 — Retrieval Strategies (Milestone M4)

**DSML stage:** modeling (retrieval). Implements and compares the three retrieval modes from the
feasibility studies on the Nvidia PoC graph:

| | Strategy | Best for | Weakness |
|---|---|---|---|
| A | **Text-to-Cypher** | rigid quantitative questions | brittle syntax, schema drift |
| B | **Vector RAG** | isolated semantic lookups | no structure, misses multi-hop |
| C | **Entity-first hybrid** *(recommended)* | supply-chain / risk intelligence | needs entity anchors in the query |

Strategy C: detect anchor entities in the query (canonical alias matching — deterministic, no LLM),
pull their 1–2-hop relation subgraph, then run vector search **scoped to EvidenceSpans that mention the
anchors** — structure + semantics combined.

In [1]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from litellm import completion

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity()

def load_embedder(name: str) -> SentenceTransformer:
    """Load from the local Hugging Face cache (no network); download only if truly absent."""
    try:
        return SentenceTransformer(name, local_files_only=True)
    except OSError:
        print(f"{name} not in the local cache — downloading once...")
        return SentenceTransformer(name)

model = load_embedder(os.getenv("EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-0.6B"))
LLM_MODEL = os.environ["LLM_MODEL"]
CANONICAL = json.loads((PROJECT_ROOT / "artifacts/canonical_entities.json").read_text())

# Queries get a model-appropriate prompt; passages were embedded plain (notebook 09).
# Qwen3-Embedding ships a built-in 'query' prompt; bge-style models need a manual prefix.
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def embed_query(q: str) -> list[float]:
    if "query" in (model.prompts or {}):
        return model.encode([q], prompt_name="query", normalize_embeddings=True)[0].tolist()
    return model.encode([BGE_QUERY_PREFIX + q], normalize_embeddings=True)[0].tolist()

def run_cypher(query: str, **params) -> list[dict]:
    with driver.session() as session:
        return [dict(r) for r in session.run(query, **params)]
print("ready")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

ready


## A. Text-to-Cypher — LLM writes the query from a schema card

In [2]:
SCHEMA_CARD = """Nodes:
  Company(cik INT UNIQUE, ticker, name, tier, sec_filer)
  Filing(accession_no UNIQUE, form, filing_date DATE, url)
  FilingSection(section_key UNIQUE, section_id, title)
  RiskFactor(risk_id UNIQUE, summary, category)
  Metric(metric_id UNIQUE, metric ['revenue','capex','rnd','net_income'], value FLOAT, unit, period_start DATE, period_end DATE)
  Product(name UNIQUE, type)
  EvidenceSpan(chunk_id UNIQUE, text, source_url)
Relationships:
  (Company)-[:FILED {date}]->(Filing)
  (Filing)-[:HAS_SECTION]->(FilingSection)
  (EvidenceSpan)-[:FROM_SECTION]->(FilingSection)
  (EvidenceSpan)-[:MENTIONS]->(Company)
  (Company)-[:REPORTS_METRIC {accession_no}]->(Metric)
  (Company)-[:DISCLOSES_RISK {start_date, status}]->(RiskFactor)
  (RiskFactor)-[:HAS_EVIDENCE {quote}]->(EvidenceSpan)
  (Company)-[:SUPPLIES_TO|DEPENDS_ON|CUSTOMER_OF|COMPETES_WITH {start_date, status, evidence_chunk_ids, evidence_quote}]->(Company)
  (Product)-[:MENTIONED_IN]->(EvidenceSpan)"""

def text_to_cypher(question: str) -> tuple[str, list[dict]]:
    prompt = (
        f"Write a single read-only Cypher query for Neo4j 5 answering the question below.\n"
        f"Schema:\n{SCHEMA_CARD}\n\nQuestion: {question}\n\n"
        f"Rules: RETURN descriptive column aliases; LIMIT 25; no writes; reply with ONLY the Cypher, no fences."
    )
    # Sonnet 5: no sampling params (400) + thinking disabled (on by default, can return content=None on capped calls)
    resp = completion(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}],
                      max_tokens=400, thinking={"type": "disabled"})
    content = resp.choices[0].message.content or ""
    cypher = re.sub(r"^```(cypher)?|```$", "", content.strip(), flags=re.MULTILINE).strip()
    try:
        return cypher, run_cypher(cypher)
    except Exception as e:
        return cypher, [{"error": str(e)[:200]}]

cypher, rows = text_to_cypher("What was Nvidia's revenue for the fiscal year ending 2024-01-28?")
print(cypher, "\n")
pd.DataFrame(rows)

MATCH (c:Company)-[:REPORTS_METRIC]->(m:Metric)
WHERE toLower(c.name) CONTAINS 'nvidia' AND m.metric = 'revenue' AND m.period_end = date('2024-01-28')
RETURN c.name AS company_name, m.value AS revenue_value, m.unit AS revenue_unit, m.period_start AS period_start, m.period_end AS period_end
LIMIT 25 



,company_name,revenue_value,revenue_unit,period_start,period_end
0,Nvidia,6.092200e+10,USD,2023-01-30,2024-01-28


## B. Plain vector RAG over EvidenceSpans

In [3]:
def vector_rag(question: str, k: int = 5) -> pd.DataFrame:
    rows = run_cypher(
        """CALL db.index.vector.queryNodes('evidence_embedding', $k, $vec) YIELD node, score
        RETURN node.chunk_id AS chunk_id, score, node.sub_heading AS sub_heading,
               left(node.text, 180) AS preview""",
        k=k, vec=embed_query(question),
    )
    return pd.DataFrame(rows)

vector_rag("How do export controls affect Nvidia's business in China?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('evidence_embedding', $k, $vec) YIELD node, score\n        RETURN node.chunk_id AS chunk_id, score, node.sub_heading AS sub_heading,\n               left(node.text, 180) AS preview"


,chunk_id,score,sub_heading,preview
0,0001045810-24-000029:I.1A:0152,0.891806,Our operations could be affected by the comple...,and its allies. The United States has imposed ...
1,0001045810-23-000017:I.1A:0051,0.871746,Our operations could be affected by the comple...,"government, or USG, announced new export restr..."
2,0001045810-25-000023:I.1A:0256,0.854609,"We are subject to complex laws, rules, regulat...","Concurrently, the war in Ukraine has impacted ..."
3,0001045810-25-000023:I.1A:0258,0.850771,"We are subject to complex laws, rules, regulat...",We expanded our Data Center product portfolio ...
4,0001045810-24-000029:I.1A:0153,0.850771,Our operations could be affected by the comple...,We are working to expand our Data Center produ...


## C. Entity-first hybrid (recommended) — anchors → subgraph → scoped vectors

In [4]:
ALIAS_TO_ID = {}
for name, spec in CANONICAL.items():
    for alias in {name, *spec["aliases"]}:
        ALIAS_TO_ID[alias.lower()] = (name, spec["entity_id"])
ALIAS_RES = [(re.compile(rf"\b{re.escape(a)}\b", re.I), v) for a, v in ALIAS_TO_ID.items()]

def detect_anchors(question: str) -> dict[str, int]:
    return {name: eid for pat, (name, eid) in ALIAS_RES if pat.search(question)}

def hybrid_retrieve(question: str, k_chunks: int = 6, hops: int = 2) -> dict:
    anchors = detect_anchors(question)
    anchor_ids = list(anchors.values()) or [1045810]  # default anchor: Nvidia (the PoC filer)
    edges = run_cypher(
        f"""MATCH (a:Company) WHERE a.cik IN $ids
        MATCH p = (a)-[r:SUPPLIES_TO|DEPENDS_ON|CUSTOMER_OF|COMPETES_WITH*1..{hops}]-(b:Company)
        UNWIND relationships(p) AS rel
        RETURN DISTINCT startNode(rel).name AS source, type(rel) AS relation, endNode(rel).name AS target,
               rel.status AS status, rel.evidence_quote AS quote, rel.evidence_chunk_ids AS chunk_ids""",
        ids=anchor_ids,
    )
    risks = run_cypher(
        """MATCH (a:Company)-[d:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)
        WHERE a.cik IN $ids
        CALL db.index.vector.queryNodes('risk_embedding', 25, $vec) YIELD node, score
        WITH rf, node, score WHERE node = rf
        RETURN rf.summary AS summary, rf.category AS category, score ORDER BY score DESC LIMIT 6""",
        ids=anchor_ids, vec=embed_query(question),
    )
    chunks = run_cypher(
        """CALL db.index.vector.queryNodes('evidence_embedding', 40, $vec) YIELD node, score
        MATCH (node)-[:MENTIONS]->(c:Company) WHERE c.cik IN $ids
        RETURN DISTINCT node.chunk_id AS chunk_id, score, node.text AS text,
               node.source_url AS source_url ORDER BY score DESC LIMIT $k""",
        ids=anchor_ids, vec=embed_query(question), k=k_chunks,
    )
    return {"anchors": anchors, "edges": edges, "risks": risks, "chunks": chunks}

result = hybrid_retrieve("How does Nvidia depend on TSMC, and what supply-chain risks does it disclose?")
print("anchors:", result["anchors"])
print(f"{len(result['edges'])} subgraph edges, {len(result['risks'])} scoped risks, {len(result['chunks'])} scoped chunks")
pd.DataFrame(result["edges"])[["source", "relation", "target", "quote"]].head(10)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=3, column=9, offset=108>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 108, 'line': 3, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a:Company)-[d:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)\n        WHERE a.cik IN $ids\n        CALL db.index.vector.queryNodes('risk_embedding', 25, $vec) YIELD node, score\n        WITH rf, node, score WHERE node = rf\n        RETURN rf.summary AS summary, rf.category AS category, score ORDER BY score DESC LIMIT 6"
Receiv

anchors: {'Nvidia': 1045810, 'TSMC': 1046179}
17 subgraph edges, 6 scoped risks, 6 scoped chunks


,source,relation,target,quote
0,Nvidia,DEPENDS_ON,TSMC,"We utilize foundries, such as Taiwan Semicondu..."
1,Nvidia,DEPENDS_ON,Samsung,"We utilize foundries, such as Taiwan Semicondu..."
2,Nvidia,CUSTOMER_OF,Samsung,"We purchase memory from SK Hynix Inc., Micron ..."
3,Nvidia,COMPETES_WITH,Samsung,"such as Ambarella, Inc., AMD, Broadcom, Intel,..."
4,Nvidia,DEPENDS_ON,Foxconn,We engage with independent subcontractors and ...
5,Nvidia,CUSTOMER_OF,Micron,"We purchase memory from SK Hynix Inc., Micron ..."
6,Nvidia,CUSTOMER_OF,SK Hynix,"We purchase memory from SK Hynix Inc., Micron ..."
7,Nvidia,COMPETES_WITH,Microsoft,"such as Alibaba Group, Alphabet Inc., Amazon, ..."
8,Nvidia,COMPETES_WITH,Amazon,"such as Alibaba Group, Alphabet Inc., Amazon, ..."
9,Nvidia,COMPETES_WITH,Alphabet,"such as Alibaba Group, Alphabet Inc., Amazon, ..."


## Side-by-side comparison on the feasibility studies' retrieval flows (Nvidia-scope subset)

In [5]:
TEST_QUERIES = [
    "Which companies does Nvidia depend on for manufacturing?",              # flow A analogue
    "How has Nvidia's AI-related risk disclosure characterized export controls?",  # flow B/F analogue
    "What evidence supports the claim that AI demand is increasing data center revenue?",  # flow G
    "Does Nvidia disclose dependency on third-party foundries?",             # flow H
]

comparison = []
for q in TEST_QUERIES:
    h = hybrid_retrieve(q)
    v = vector_rag(q, k=6)
    comparison.append({
        "question": q,
        "anchors": ", ".join(h["anchors"]) or "(default NVDA)",
        "hybrid_edges": len(h["edges"]),
        "hybrid_chunks": len(h["chunks"]),
        "vector_only_chunks": len(v),
    })
pd.DataFrame(comparison)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=3, column=9, offset=108>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 108, 'line': 3, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a:Company)-[d:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)\n        WHERE a.cik IN $ids\n        CALL db.index.vector.queryNodes('risk_embedding', 25, $vec) YIELD node, score\n        WITH rf, node, score WHERE node = rf\n        RETURN rf.summary AS summary, rf.category AS category, score ORDER BY score DESC LIMIT 6"
Receiv

,question,anchors,hybrid_edges,hybrid_chunks,vector_only_chunks
0,Which companies does Nvidia depend on for manu...,Nvidia,17,6,6
1,How has Nvidia's AI-related risk disclosure ch...,Nvidia,17,6,6
2,What evidence supports the claim that AI deman...,(default NVDA),17,6,6
3,Does Nvidia disclose dependency on third-party...,Nvidia,17,6,6


In [6]:
# --- M4 (retrieval) assertion cell ---
r = hybrid_retrieve("How does Nvidia depend on TSMC?")
assert set(r["anchors"]) >= {"Nvidia", "TSMC"}, f"anchor detection failed: {r['anchors']}"
assert len(r["edges"]) >= 1, "no subgraph edges for Nvidia-TSMC"
assert any("TSMC" in (e["source"], e["target"]) for e in r["edges"])
assert len(r["chunks"]) >= 3 and all(c["source_url"] for c in r["chunks"])
assert len(vector_rag("export controls China")) == 5
driver.close()
print("M4 (retrieval) OK — all three strategies functional; hybrid returns structure + scoped evidence")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=3, column=9, offset=108>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 108, 'line': 3, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a:Company)-[d:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)\n        WHERE a.cik IN $ids\n        CALL db.index.vector.queryNodes('risk_embedding', 25, $vec) YIELD node, score\n        WITH rf, node, score WHERE node = rf\n        RETURN rf.summary AS summary, rf.category AS category, score ORDER BY score DESC LIMIT 6"
Receiv

M4 (retrieval) OK — all three strategies functional; hybrid returns structure + scoped evidence
